In [ ]:
import time
from IPython.display import Javascript
display(Javascript('''
  function ClickConnect(){
    document.querySelector('#top-toolbar > colab-connect-button').click()
  }
  setInterval(ClickConnect, 60000)
'''))
print('Anti-disconnect activated')


<IPython.core.display.Javascript object>

Anti-disconnect activated


In [ ]:
!pip install ultralytics roboflow -q
!pip install onnx onnxruntime -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.5/169.5 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 77.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 130.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 92.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 20.9 MB/s eta 0:00:00


In [ ]:

import os
from ultralytics import YOLO
print("✅ Ultralytics ready, CUDA:", os.popen("nvidia-smi --query-gpu=name --format=csv,noheader").read().strip())

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✅ Ultralytics ready, CUDA: Tesla T4


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving archive.zip to archive.zip


In [ ]:
# Cell 3 — Extract the main zipped dataset
import zipfile

with zipfile.ZipFile('/content/archive.zip', 'r') as z:
    z.extractall('/content/pothole_data')
print("✅ archive.zip extracted successfully")


✅ archive.zip extracted successfully


In [ ]:
# Cell 4 — Setup master directory structure for merged datasets
import os, shutil
from pathlib import Path

merged_train_img = '/content/merged/train/images'
merged_train_lbl = '/content/merged/train/labels'
merged_val_img   = '/content/merged/valid/images'
merged_val_lbl   = '/content/merged/valid/labels'

for p in [merged_train_img, merged_train_lbl, merged_val_img, merged_val_lbl]:
    os.makedirs(p, exist_ok=True)
print("✅ Master directories created")


✅ Master directories created


In [ ]:
# Cell 5 — Copy all images and labels into the merged folder (Bulletproof Search)
# List of directories containing datasets (Assuming they are available in /content/)
DATASETS = [
    '/content/pothole_data',          # road_damage_2025
    '/content/sachin_patel',          # sachin_patel dataset (if uploaded)
    '/content/andrew_mvd',            # andrew_mvd dataset (if uploaded)
]

for ds in DATASETS:
    if Path(ds).exists():
        # Recursively find every .jpg image no matter what folder it's hiding in
        for img in Path(ds).rglob('*.jpg'):
            shutil.copy(img, merged_train_img)

        # Recursively find every YOLO label .txt (ignoring instructions/classes files)
        for lbl in Path(ds).rglob('*.txt'):
            if lbl.name not in ['classes.txt', 'readme.txt']:
                shutil.copy(lbl, merged_train_lbl)

print('✅ Merging complete. Total train images:', len(list(Path(merged_train_img).glob('*.jpg'))))


✅ Merging complete. Total train images: 2009


In [ ]:
# Cell 6 — Create data.yaml mapping to our 3 classes
data_yaml = """
train: /content/merged/train/images
val: /content/merged/train/images  # Using the same set for validation for speed

nc: 3
names: ['pothole', 'crack', 'manhole']
"""

with open('/content/pothole.yaml', 'w') as f:
    f.write(data_yaml)

print("✅ data.yaml written")


✅ data.yaml written


In [ ]:
# Cell 7 — Train (45-60 min on T4 GPU)
model = YOLO('yolov8n.pt')

results = model.train(
    data='/content/pothole.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    name='pothole_v1',
    project='/content/runs/detect',
    device=0,
    save=True,
)
print("✅ YOLO Training Finished")


Ultralytics 8.4.37 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/pothole.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=pothole_v12, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, pers

In [ ]:
# Cell 8 — Export ONNX
best_model = YOLO('/content/runs/detect/pothole_v1/weights/best.pt')
best_model.export(format='onnx', imgsz=640, opset=12, simplify=True)
best_model.export(format='torchscript')
print("✅ ONNX models generated")


Ultralytics 8.4.37 🚀 Python-3.12.13 torch-2.10.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
Model summary (fused): 73 layers, 3,006,233 parameters, 0 gradients, 8.1 GFLOPs

PyTorch: starting from '/content/runs/detect/pothole_v1/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 7, 8400) (5.9 MB)
requirements: Ultralytics requirements ['onnxslim>=0.1.71', 'onnxruntime-gpu'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 12 packages in 446ms
Prepared 3 packages in 4.89s
Installed 3 packages in 39ms
 + colorama==0.4.6
 + onnxruntime-gpu==1.24.4
 + onnxslim==0.1.91

requirements: AutoUpdate success ✅ 6.3s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.21.0 opset 12...
ONNX: slimming with onnxslim 0.1.91...
ONNX: ex

In [ ]:
# Cell 9 — Download the weights
import shutil
from google.colab import files
shutil.make_archive('/content/pothole_weights', 'zip', '/content/runs/detect/pothole_v1/weights')
files.download('/content/pothole_weights.zip')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>